                    User embedding
                         │
              ┌──────────┴──────────┐
              ↓                     ↓
       Semantic retrieval      Popularity /
          Top 100               exploration
              │                     │
              └──────────┬──────────┘
                         ↓
                  Candidate pool
                     Top 300
                         │
                  Remove seen items
                         │
                         ↓
                Candidate features
                         │
             ┌───────────┴───────────┐
             ↓                       ↓
       User-item similarity      Domain match
             ↓                       ↓
       BPR neural score         Other signals
             └───────────┬───────────┘
                         ↓
                    Final score
                         ↓
                       Top 10

Phase 2

   ↓

User embeddings

   ↓

Candidate retrieval

   ↓

Candidate Recall check

   ↓

Create positive/negative training pairs

   ↓

BPR model

   ↓

Train

   ↓

Retrieve candidates for each user

   ↓

BPR re-rank

   ↓

Top 10 predictions

   ↓

Exact + Semantic evaluation

   ↓
   
Save CSV

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import faiss

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
BASE_PATH = r"C:\nihal\rough\Reccomender_system_rough"

DATA_PATH = os.path.join(
    BASE_PATH,
    "Cleaned_dataset"
)

PHASE2_PATH = os.path.join(
    BASE_PATH,
    "Phase_2"
)

PHASE3_PATH = os.path.join(
    BASE_PATH,
    "Phase_3"
)

os.makedirs(
    PHASE3_PATH,
    exist_ok=True
)

print("DATA_PATH:", DATA_PATH)
print("PHASE2_PATH:", PHASE2_PATH)
print("PHASE3_PATH:", PHASE3_PATH)

DATA_PATH: C:\nihal\rough\Reccomender_system_rough\Cleaned_dataset
PHASE2_PATH: C:\nihal\rough\Reccomender_system_rough\Phase_2
PHASE3_PATH: C:\nihal\rough\Reccomender_system_rough\Phase_3


In [ ]:
items = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "items.csv"
    )
)

print("Items shape:", items.shape)
print(items.columns.tolist())

Items shape: (8787, 12)
['id', 'source', 'title', 'domain', 'metadata', 'description', 'release_year', 'duration', 'rating', 'language', 'country', 'combined']


In [ ]:
movie_items = items[
    items['source'].astype(str).str.lower() == 'movies'
].copy()

movie_items = movie_items.reset_index(
    drop=True
)

print("Movie items:", len(movie_items))
print(movie_items.head())

Movie items: 1000
           id  source            title           domain          metadata  \
0  movie_0001  movies    dragon legend  stand-up comedy  history thriller   
1  movie_0002  movies    storm warrior  stand-up comedy            sci-fi   
2  movie_0003  movies      fire family            movie             drama   
3  movie_0004  movies     our princess      documentary            sci-fi   
4  movie_0005  movies  warrior mission      documentary     sport mystery   

  description  release_year duration rating  language country  \
0         NaN          2014     35.0   TV-Y    french   japan   
1         NaN          2017     37.0     PG  japanese     usa   
2         NaN          2003    142.0  TV-MA   english     usa   
3         NaN          2011    131.0  NC-17  japanese     usa   
4         NaN          2015     91.0   TV-G   english     usa   

                                            combined  
0  Title: dragon legend Domain: stand-up comedy M...  
1  Title: storm wa

In [ ]:
item_embeddings = torch.load(
    os.path.join(
        DATA_PATH,
        "item_embeddings.pt"
    ),
    map_location="cpu"
)

print(
    "Item embeddings:",
    item_embeddings.shape
)

Item embeddings: torch.Size([8787, 384])


C:\Users\nihal\AppData\Local\Temp\ipykernel_28992\2500224100.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  item_embeddings = torch.load(


In [ ]:
movie_indices = items[
    items['source'].astype(str).str.lower() == 'movies'
].index

movie_embeddings = item_embeddings[
    movie_indices
]

movie_embeddings = movie_embeddings.float()

print(
    "Movie embeddings:",
    movie_embeddings.shape
)

Movie embeddings: torch.Size([1000, 384])


In [ ]:
movie_id_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(
        movie_items['id']
    )
}

print(
    "Movie ID mappings:",
    len(movie_id_to_idx)
)

Movie ID mappings: 1000


In [ ]:
assert len(movie_items) == movie_embeddings.shape[0]

assert len(movie_id_to_idx) == len(movie_items)

assert movie_items['id'].is_unique

print(
    "Movie embedding mapping verified."
)

Movie embedding mapping verified.


In [ ]:
movie_history = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "movie_history.csv"
    )
)

print(
    "Movie history:",
    movie_history.shape
)

print(
    movie_history.columns.tolist()
)

Movie history: (105000, 22)
['session_id', 'user_id', 'item_id', 'interaction_date', 'device_type', 'time', 'progress_percentage', 'action', 'quality', 'location_country', 'is_download', 'user_rating', 'time_capped', 'time_norm', 'progress_norm', 'action_score', 'rating_score', 'engagement_score', 'title', 'combined', 'metadata', 'domain']


In [ ]:
train_interactions = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "movie_train_interactions.csv"
    )
)

print("Train:", train_interactions.shape)
print(train_interactions.head())

Train: (89470, 4)
      user_id     item_id interaction_date  engagement_score
0  user_00001  movie_0693       2024-02-05          0.207000
1  user_00001  movie_0672       2024-05-06          0.391500
2  user_00001  movie_0125       2024-05-22          0.678402
3  user_00001  movie_0450       2024-06-22          0.511926
4  user_00001  movie_0665       2024-07-22          0.543260


In [ ]:
test_interactions = pd.read_csv(
    os.path.join(
        DATA_PATH,
        "movie_test_interactions.csv"
    )
)

print("Test:", test_interactions.shape)
print(test_interactions.head())

Test: (10000, 4)
      user_id     item_id interaction_date  engagement_score
0  user_00001  movie_0483       2025-10-30          0.585000
1  user_00002  movie_0362       2025-11-28          0.240696
2  user_00003  movie_0855       2025-06-24          0.321370
3  user_00004  movie_0712       2025-12-29          0.581564
4  user_00005  movie_0423       2025-09-19          0.367155


In [ ]:
train_users = set(
    train_interactions['user_id']
)

test_users = set(
    test_interactions['user_id']
)

print(
    "Train users:",
    len(train_users)
)

print(
    "Test users:",
    len(test_users)
)

print(
    "Users present in both:",
    len(train_users & test_users)
)

Train users: 10000
Test users: 10000
Users present in both: 10000


In [ ]:
valid_movie_ids = set(
    movie_items['id']
)

invalid_train = (
    ~train_interactions['item_id'].isin(
        valid_movie_ids
    )
).sum()

invalid_test = (
    ~test_interactions['item_id'].isin(
        valid_movie_ids
    )
).sum()

print(
    "Invalid train movie IDs:",
    invalid_train
)

print(
    "Invalid test movie IDs:",
    invalid_test
)

Invalid train movie IDs: 0
Invalid test movie IDs: 0


In [ ]:
user_embeddings = torch.load(
    os.path.join(
        PHASE2_PATH,
        "user_embeddings.pt"
    ),
    map_location="cpu"
)

print(
    "Number of users:",
    len(user_embeddings)
)

C:\Users\nihal\AppData\Local\Temp\ipykernel_28992\264282379.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  user_embeddings = torch.load(


Number of users: 10000


In [ ]:
first_user = list(
    user_embeddings.keys()
)[0]

print(
    "First user:",
    first_user
)

print(
    "Embedding shape:",
    user_embeddings[first_user].shape
)

First user: user_00001
Embedding shape: torch.Size([384])


In [ ]:
movie_embeddings = movie_embeddings.to(
    device
)

for user_id in user_embeddings:

    user_embeddings[user_id] = torch.tensor(
        user_embeddings[user_id],
        dtype=torch.float32
    ).to(device)

print("Embeddings moved to:", device)

C:\Users\nihal\AppData\Local\Temp\ipykernel_28992\2828059185.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  user_embeddings[user_id] = torch.tensor(


Embeddings moved to: cuda


In [ ]:
index = faiss.IndexFlatIP(
    movie_embeddings.shape[1]
)

movie_embeddings_cpu = (
    movie_embeddings
    .detach()
    .cpu()
    .numpy()
    .astype('float32')
)

index.add(
    movie_embeddings_cpu
)

print(
    "FAISS items:",
    index.ntotal
)

FAISS items: 1000


In [ ]:
user_seen_movies = {}

for user_id in train_interactions[
    'user_id'
].unique():

    user_rows = train_interactions[
        train_interactions['user_id'] == user_id
    ]

    user_seen_movies[user_id] = set(
        user_rows['item_id'].tolist()
    )

print(
    "Users with history:",
    len(user_seen_movies)
)

Users with history: 10000


In [ ]:
print(
    "Movie items:",
    len(movie_items)
)

print(
    "Movie embeddings:",
    movie_embeddings.shape
)

print(
    "Train interactions:",
    len(train_interactions)
)

print(
    "Test interactions:",
    len(test_interactions)
)

print(
    "User embeddings:",
    len(user_embeddings)
)

print(
    "FAISS items:",
    index.ntotal
)

Movie items: 1000
Movie embeddings: torch.Size([1000, 384])
Train interactions: 89470
Test interactions: 10000
User embeddings: 10000
FAISS items: 1000


In [ ]:
def retrieve_candidates_with_scores(
    user_id,
    candidate_k=100
):

    if user_id not in user_embeddings:
        return []

    # Get Phase-2 user embedding
    user_vec = (
        user_embeddings[user_id]
        .detach()
        .cpu()
        .numpy()
        .astype("float32")
        .reshape(1, -1)
    )

    # Get extra candidates because
    # some may already be watched
    search_k = min(
        candidate_k + len(
            user_seen_movies.get(
                user_id,
                set()
            )
        ),
        index.ntotal
    )

    scores, indices = index.search(
        user_vec,
        search_k
    )

    seen_movies = user_seen_movies.get(
        user_id,
        set()
    )

    candidates = []

    for i in range(
        len(indices[0])
    ):

        idx = indices[0][i]

        if idx < 0:
            continue

        movie_id = movie_items.iloc[
            idx
        ]["id"]

        # Don't recommend already watched movies
        if movie_id in seen_movies:
            continue

        candidates.append({
            "movie_id": movie_id,
            "semantic_score": float(
                scores[0][i]
            )
        })

        if len(candidates) >= candidate_k:
            break

    return candidates

In [ ]:
user_id = test_interactions.iloc[0]["user_id"]

candidates = retrieve_candidates_with_scores(
    user_id,
    candidate_k=100
)

print(
    "User:",
    user_id
)

print(
    "Number of candidates:",
    len(candidates)
)

print(
    "Candidates:"
)

print(candidates[:20])

User: user_00001
Number of candidates: 100
Candidates:
['movie_0159', 'movie_0361', 'movie_0840', 'movie_0592', 'movie_0189', 'movie_0003', 'movie_0916', 'movie_0611', 'movie_0431', 'movie_0789', 'movie_0374', 'movie_0445', 'movie_0292', 'movie_0314', 'movie_0475', 'movie_0662', 'movie_0109', 'movie_0712', 'movie_0133', 'movie_0100']


In [ ]:
actual_items = test_interactions[
    test_interactions["user_id"] == user_id
]["item_id"].tolist()

print(
    "Actual test items:",
    actual_items
)

Actual test items: ['movie_0483']


In [ ]:
for actual_item in actual_items:

    if actual_item in candidates:

        rank = (
            candidates.index(actual_item) + 1
        )

        print(
            actual_item,
            "FOUND at candidate rank",
            rank
        )

    else:

        print(
            actual_item,
            "NOT FOUND"
        )

movie_0483 FOUND at candidate rank 59


In [ ]:
def candidate_recall_at_k(
    k
):
    hits = 0
    evaluated = 0

    for user_id in test_users:

        if user_id not in user_embeddings:
            continue

        relevant = test_interactions[
            test_interactions["user_id"] == user_id
        ]["item_id"].tolist()

        if len(relevant) == 0:
            continue

        candidates = retrieve_candidates_with_scores(
            user_id,
            candidate_k=k
        )

        if (
            len(
                set(relevant) &
                set(candidates)
            ) > 0
        ):
            hits += 1

        evaluated += 1

    if evaluated == 0:
        return 0.0

    return hits / evaluated

In [ ]:
candidate_recall_10 = (
    candidate_recall_at_k(10)
)

candidate_recall_50 = (
    candidate_recall_at_k(50)
)

candidate_recall_100 = (
    candidate_recall_at_k(100)
)

candidate_recall_500 = (
    candidate_recall_at_k(500)
)

candidate_recall_1000 = (
    candidate_recall_at_k(1000)
)

print(
    "Candidate Recall@10:",
    candidate_recall_10
)

print(
    "Candidate Recall@50:",
    candidate_recall_50
)

print(
    "Candidate Recall@100:",
    candidate_recall_100
)

print(
    "Candidate Recall@500:",
    candidate_recall_500
)

print(
    "Candidate Recall@1000:",
    candidate_recall_1000
)

Candidate Recall@10: 0.0111
Candidate Recall@50: 0.0502
Candidate Recall@100: 0.1016
Candidate Recall@500: 0.5001
Candidate Recall@1000: 1.0


In [ ]:
candidate_metrics = pd.DataFrame([
    {
        "model": "Phase 2 Candidate Retrieval",
        "CandidateRecall@10":
            candidate_recall_10,
        "CandidateRecall@50":
            candidate_recall_50,
        "CandidateRecall@100":
            candidate_recall_100,
        "CandidateRecall@500":
            candidate_recall_500,
        "CandidateRecall@1000":
            candidate_recall_1000
    }
])

candidate_metrics

,model,CandidateRecall@10,CandidateRecall@50,CandidateRecall@100,CandidateRecall@500,CandidateRecall@1000
0,Phase 2 Candidate Retrieval,0.0111,0.0502,0.1016,0.5001,1.0


In [ ]:
candidate_metrics.to_csv(
    os.path.join(
        PHASE3_PATH,
        "phase3_candidate_metrics.csv"
    ),
    index=False
)

print(
    "Candidate metrics saved."
)

Candidate metrics saved.


In [ ]:
candidate_counts = []

for user_id in list(test_users)[:100]:

    candidates = retrieve_candidates_with_scores(
        user_id,
        candidate_k=100
    )

    candidate_counts.append(
        len(candidates)
    )

print(
    "Average candidates:",
    np.mean(candidate_counts)
)

print(
    "Minimum candidates:",
    np.min(candidate_counts)
)

print(
    "Maximum candidates:",
    np.max(candidate_counts)
)

Average candidates: 100.0
Minimum candidates: 100
Maximum candidates: 100


In [ ]:
sample_user = list(test_users)[0]

sample_candidates = retrieve_candidates_with_scores(
    sample_user,
    candidate_k=20
)

sample_actual = test_interactions[
    test_interactions["user_id"] == sample_user
]["item_id"].tolist()

print("User:", sample_user)

print(
    "Actual:",
    sample_actual
)

print(
    "Top 20 candidates:"
)

for rank, movie_id in enumerate(
    sample_candidates,
    start=1
):

    title = movie_items[
        movie_items["id"] == movie_id
    ]["title"].iloc[0]

    print(
        rank,
        movie_id,
        "->",
        title
    )

User: user_01838
Actual: ['movie_0031']
Top 20 candidates:
1 movie_0996 -> secret mission
2 movie_0718 -> mystery dream
3 movie_0951 -> big house
4 movie_0564 -> mystery mystery
5 movie_0577 -> family house
6 movie_0020 -> a mystery
7 movie_0073 -> house family
8 movie_0068 -> secret night
9 movie_0572 -> dream secret
10 movie_0345 -> dark secret
11 movie_0883 -> a mission
12 movie_0458 -> secret hero
13 movie_0059 -> night secret
14 movie_0876 -> mystery war
15 movie_0847 -> mystery war
16 movie_0882 -> little mystery
17 movie_0032 -> house hero
18 movie_0358 -> legend secret
19 movie_0200 -> legend secret
20 movie_0858 -> fire mystery


In [ ]:
train_data = train_interactions.copy()

train_data["movie_idx"] = train_data["item_id"].map(
    movie_id_to_idx
)

train_data = train_data.dropna(
    subset=["movie_idx"]
)

train_data["movie_idx"] = (
    train_data["movie_idx"].astype(int)
)

print(
    "Training rows:",
    len(train_data)
)

print(
    train_data[
        ["user_id", "item_id", "movie_idx", "engagement_score"]
    ].head()
)

Training rows: 89470
      user_id     item_id  movie_idx  engagement_score
0  user_00001  movie_0693        692          0.207000
1  user_00001  movie_0672        671          0.391500
2  user_00001  movie_0125        124          0.678402
3  user_00001  movie_0450        449          0.511926
4  user_00001  movie_0665        664          0.543260


In [ ]:
user_positive_movies = {}

for _, row in train_data.iterrows():

    user_id = row["user_id"]

    movie_id = row["item_id"]

    if user_id not in user_positive_movies:

        user_positive_movies[user_id] = set()

    user_positive_movies[user_id].add(
        movie_id
    )

print(
    "Users:",
    len(user_positive_movies)
)

NameError: name 'train_data' is not defined

In [ ]:
import random

def get_hard_negatives(
    user_id,
    candidate_k=100,
    num_negatives=5
):

    candidates = retrieve_candidates_with_scores(
        user_id,
        candidate_k=candidate_k
    )

    if len(candidates) == 0:
        return []

    hard_pool_size = min(
        50,
        len(candidates)
    )

    hard_pool = candidates[
        :hard_pool_size
    ]

    if len(hard_pool) <= num_negatives:

        return [
            x["movie_id"]
            for x in hard_pool
        ]

    selected = random.sample(
        hard_pool,
        num_negatives
    )

    return [
        x["movie_id"]
        for x in selected
    ]

In [ ]:
sample_user = train_data.iloc[0]["user_id"]

sample_positive = train_data.iloc[0]["item_id"]

hard_negatives = get_hard_negatives(
    sample_user,
    candidate_k=100,
    num_negatives=5
)

print(
    "User:",
    sample_user
)

print(
    "Positive:",
    sample_positive
)

print(
    "Hard negatives:"
)

for movie_id in hard_negatives:

    print(movie_id)

Unique positive interactions: 89470


In [ ]:
bpr_samples = []

for count, (_, row) in enumerate(
    train_data.iterrows()
):

    user_id = row["user_id"]

    positive_movie = row["item_id"]

    hard_negatives = get_hard_negatives(
        user_id,
        candidate_k=100,
        num_negatives=5
    )

    for negative_movie in hard_negatives:

        bpr_samples.append({
            "user_id": user_id,
            "positive_movie": positive_movie,
            "negative_movie": negative_movie
        })

    if count % 10000 == 0:

        print(
            "Processed:",
            count
        )

print(
    "Total BPR samples:",
    len(bpr_samples)
)

Users: 10000


In [ ]:
bpr_samples_df = pd.DataFrame(
    bpr_samples
)

print(
    bpr_samples_df.shape
)

print(
    bpr_samples_df.head()
)

User: 2773
Negative movie index: 3
Is actually positive? False


In [ ]:
print(
    "Unique users:",
    bpr_samples_df[
        "user_id"
    ].nunique()
)

print(
    "Unique positive movies:",
    bpr_samples_df[
        "positive_movie"
    ].nunique()
)

print(
    "Unique negative movies:",
    bpr_samples_df[
        "negative_movie"
    ].nunique()
)

In [ ]:
user_vectors = []

positive_vectors = []

negative_vectors = []

for _, row in bpr_samples_df.iterrows():

    user_id = row["user_id"]

    positive_movie = row[
        "positive_movie"
    ]

    negative_movie = row[
        "negative_movie"
    ]

    user_vector = user_embeddings[
        user_id
    ]

    positive_idx = movie_id_to_idx[
        positive_movie
    ]

    negative_idx = movie_id_to_idx[
        negative_movie
    ]

    positive_vector = movie_embeddings[
        positive_idx
    ]

    negative_vector = movie_embeddings[
        negative_idx
    ]

    user_vectors.append(
        user_vector
    )

    positive_vectors.append(
        positive_vector
    )

    negative_vectors.append(
        negative_vector
    )

In [ ]:
user_tensor = torch.stack(
    user_vectors
).float()

positive_tensor = torch.stack(
    positive_vectors
).float()

negative_tensor = torch.stack(
    negative_vectors
).float()

print(
    "User tensor:",
    user_tensor.shape
)

print(
    "Positive tensor:",
    positive_tensor.shape
)

print(
    "Negative tensor:",
    negative_tensor.shape
)

In [ ]:
class BPRModel(nn.Module):

    def __init__(
        self,
        embedding_dim
    ):

        super().__init__()

        self.user_layer = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.movie_layer = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.user_bias = nn.Parameter(
            torch.zeros(1)
        )

        self.movie_bias = nn.Parameter(
            torch.zeros(1)
        )

    def score(
        self,
        user_embedding,
        movie_embedding
    ):

        user_vec = self.user_layer(
            user_embedding
        )

        movie_vec = self.movie_layer(
            movie_embedding
        )

        score = (
            user_vec * movie_vec
        ).sum(dim=1)

        score += self.user_bias
        score += self.movie_bias

        return score

    def forward(
        self,
        user_embedding,
        positive_embedding,
        negative_embedding
    ):

        positive_score = self.score(
            user_embedding,
            positive_embedding
        )

        negative_score = self.score(
            user_embedding,
            negative_embedding
        )

        return (
            positive_score,
            negative_score
        )

In [ ]:
EMBEDDING_DIM = movie_embeddings.shape[1]

bpr_model = BPRModel(
    EMBEDDING_DIM
).to(device)

print(
    "Embedding dimension:",
    EMBEDDING_DIM
)

print(
    bpr_model
)

BPRModel(
  (user_embedding): Embedding(10000, 64)
  (movie_embedding): Embedding(1000, 64)
  (user_bias): Embedding(10000, 1)
  (movie_bias): Embedding(1000, 1)
)


In [ ]:
def bpr_loss(
    positive_scores,
    negative_scores
):

    difference = (
        positive_scores -
        negative_scores
    )

    loss = -torch.mean(
        torch.log(
            torch.sigmoid(
                difference
            ) + 1e-8
        )
    )

    return loss

In [ ]:
optimizer = optim.Adam(
    bpr_model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 1e-05
)


In [ ]:
EPOCHS = 10

BATCH_SIZE = 1024

num_samples = len(
    user_tensor
)

print(
    "Samples:",
    num_samples
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Epochs:",
    EPOCHS
)

Samples: 89470
Batch size: 1024
Epochs: 10


In [ ]:
for epoch in range(
    EPOCHS
):

    bpr_model.train()

    indices = torch.randperm(
        num_samples
    )

    total_loss = 0.0

    for start in range(
        0,
        num_samples,
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            num_samples
        )

        batch_indices = indices[
            start:end
        ]

        batch_users = user_tensor[
            batch_indices
        ].to(device)

        batch_positive = positive_tensor[
            batch_indices
        ].to(device)

        batch_negative = negative_tensor[
            batch_indices
        ].to(device)

        optimizer.zero_grad()

        positive_scores, negative_scores = (
            bpr_model(
                batch_users,
                batch_positive,
                batch_negative
            )
        )

        loss = bpr_loss(
            positive_scores,
            negative_scores
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() *
            len(batch_indices)
        )

    average_loss = (
        total_loss /
        num_samples
    )

    print(
        "Epoch",
        epoch + 1,
        "/",
        EPOCHS,
        "Loss:",
        average_loss
    )

Epoch 1 / 10 Loss: 0.6931893014476141
Epoch 2 / 10 Loss: 0.6899988737108456
Epoch 3 / 10 Loss: 0.6778030855730017
Epoch 4 / 10 Loss: 0.6481438479903717
Epoch 5 / 10 Loss: 0.6006669814391843
Epoch 6 / 10 Loss: 0.5408711608753692
Epoch 7 / 10 Loss: 0.47616133817123996
Epoch 8 / 10 Loss: 0.41294425338276863
Epoch 9 / 10 Loss: 0.35540873774659293
Epoch 10 / 10 Loss: 0.30544169111079683


In [ ]:
# sample_user = test_interactions.iloc[0]["user_id"]

# sample_movie = movie_items.iloc[0]["id"]

# score = get_bpr_score(
#     sample_user,
#     sample_movie
# )

# print("User:", sample_user)
# print("Movie:", sample_movie)
# print("BPR score:", score)

User: user_00001
Movie: movie_0001
BPR score: -0.15082210302352905


In [ ]:
def get_bpr_score(
    user_id,
    movie_id
):

    if user_id not in user_embeddings:
        return 0.0

    if movie_id not in movie_id_to_idx:
        return 0.0

    user_vector = user_embeddings[
        user_id
    ]

    movie_idx = movie_id_to_idx[
        movie_id
    ]

    movie_vector = movie_embeddings[
        movie_idx
    ]

    user_vector = user_vector.to(
        device
    )

    movie_vector = movie_vector.to(
        device
    )

    bpr_model.eval()

    with torch.no_grad():

        score = bpr_model.score(
            user_vector.unsqueeze(0),
            movie_vector.unsqueeze(0)
        )

    return score.item()

In [ ]:
sample_user = test_interactions.iloc[0][
    "user_id"
]

sample_movie = movie_items.iloc[0][
    "id"
]

score = get_bpr_score(
    sample_user,
    sample_movie
)

print(
    "User:",
    sample_user
)

print(
    "Movie:",
    sample_movie
)

print(
    "BPR score:",
    score
)

movie_0159 Semantic: 0.9102553129196167
movie_0361 Semantic: 0.9048712253570557
movie_0840 Semantic: 0.8995966911315918
movie_0592 Semantic: 0.8995251655578613
movie_0189 Semantic: 0.891826331615448
movie_0003 Semantic: 0.886919379234314
movie_0916 Semantic: 0.8857159614562988
movie_0611 Semantic: 0.8845853805541992
movie_0431 Semantic: 0.8837766647338867
movie_0789 Semantic: 0.8833678960800171


In [ ]:
def add_bpr_scores(
    user_id,
    candidates
):

    results = []

    for candidate in candidates:

        movie_id = candidate["movie_id"]

        semantic_score = candidate[
            "semantic_score"
        ]

        bpr_score = get_bpr_score(
            user_id,
            movie_id
        )

        results.append({
            "movie_id": movie_id,
            "semantic_score": semantic_score,
            "bpr_score": bpr_score
        })

    return results

In [ ]:
def retrieve_candidates_with_scores(
    user_id,
    candidate_k=100
):

    if user_id not in user_embeddings:
        return []

    # Get Phase-2 user embedding
    user_vec = (
        user_embeddings[user_id]
        .detach()
        .cpu()
        .numpy()
        .astype("float32")
        .reshape(1, -1)
    )

    # Get extra candidates because
    # some may already be watched
    search_k = min(
        candidate_k + len(
            user_seen_movies.get(
                user_id,
                set()
            )
        ),
        index.ntotal
    )

    scores, indices = index.search(
        user_vec,
        search_k
    )

    seen_movies = user_seen_movies.get(
        user_id,
        set()
    )

    candidates = []

    for i in range(
        len(indices[0])
    ):

        idx = indices[0][i]

        if idx < 0:
            continue

        movie_id = movie_items.iloc[
            idx
        ]["id"]

        # Don't recommend already watched movies
        if movie_id in seen_movies:
            continue

        candidates.append({
            "movie_id": movie_id,
            "semantic_score": float(
                scores[0][i]
            )
        })

        if len(candidates) >= candidate_k:
            break

    return candidates

In [ ]:
sample_user = test_users[0]

sample_candidates = retrieve_candidates_with_scores(
    sample_user,
    candidate_k=10
)

for candidate in sample_candidates:

    print(
        candidate["movie_id"],
        "->",
        candidate["semantic_score"]
    )

In [ ]:
candidates = retrieve_candidates_with_scores(
    sample_user,
    candidate_k=10
)

candidates = add_bpr_scores(
    sample_user,
    candidates
)

for candidate in candidates:

    print(
        candidate["movie_id"],
        "Semantic:",
        round(
            candidate["semantic_score"],
            4
        ),
        "BPR:",
        round(
            candidate["bpr_score"],
            4
        )
    )

movie_0159 Semantic: 0.9103 BPR: -53.3279
movie_0361 Semantic: 0.9049 BPR: -53.7969
movie_0840 Semantic: 0.8996 BPR: -53.1301
movie_0592 Semantic: 0.8995 BPR: -54.9975
movie_0189 Semantic: 0.8918 BPR: -53.0115
movie_0003 Semantic: 0.8869 BPR: -52.2107
movie_0916 Semantic: 0.8857 BPR: -54.2431
movie_0611 Semantic: 0.8846 BPR: -53.159
movie_0431 Semantic: 0.8838 BPR: -53.6583
movie_0789 Semantic: 0.8834 BPR: -53.5692


In [ ]:
def normalize_scores(
    scores
):

    if len(scores) == 0:
        return []

    min_score = min(scores)

    max_score = max(scores)

    if max_score == min_score:

        return [
            0.5
            for _ in scores
        ]

    normalized = []

    for score in scores:

        value = (
            score - min_score
        ) / (
            max_score - min_score
        )

        normalized.append(value)

    return normalized

In [ ]:
SEMANTIC_WEIGHT = 0.7
BPR_WEIGHT = 0.3

In [ ]:
def combine_scores(
    candidates
):

    bpr_scores = []

    for candidate in candidates:

        bpr_scores.append(
            candidate["bpr_score"]
        )

    normalized_bpr = normalize_scores(
        bpr_scores
    )

    results = []

    for i in range(
        len(candidates)
    ):

        semantic_score = candidates[i][
            "semantic_score"
        ]

        bpr_score = normalized_bpr[i]

        final_score = (
            SEMANTIC_WEIGHT *
            semantic_score
            +
            BPR_WEIGHT *
            bpr_score
        )

        results.append({
            "movie_id":
                candidates[i]["movie_id"],

            "semantic_score":
                semantic_score,

            "bpr_score":
                candidates[i]["bpr_score"],

            "normalized_bpr_score":
                bpr_score,

            "final_score":
                final_score
        })

    return results

In [ ]:
def rerank_candidates(
    candidates,
    top_k=10
):

    candidates = sorted(
        candidates,
        key=lambda x: x["final_score"],
        reverse=True
    )

    return candidates[:top_k]

In [ ]:
sample_candidates = retrieve_candidates_with_scores(
    sample_user,
    candidate_k=500
)

sample_candidates = add_bpr_scores(
    sample_user,
    sample_candidates
)

sample_candidates = combine_scores(
    sample_candidates
)

top_10 = rerank_candidates(
    sample_candidates,
    top_k=10
)

for rank in range(
    len(top_10)
):

    item = top_10[rank]

    print(
        "Rank",
        rank + 1,
        ":",
        item["movie_id"],
        "| Semantic =",
        round(
            item["semantic_score"],
            4
        ),
        "| BPR =",
        round(
            item["bpr_score"],
            4
        ),
        "| Final =",
        round(
            item["final_score"],
            4
        )
    )

Rank 1 : movie_0534 | Semantic = 0.7689 | BPR = -42.5427 | Final = 0.8382
Rank 2 : movie_0103 | Semantic = 0.8091 | BPR = -45.1619 | Final = 0.8033
Rank 3 : movie_0348 | Semantic = 0.7873 | BPR = -44.6191 | Final = 0.8011
Rank 4 : movie_0681 | Semantic = 0.7825 | BPR = -44.701 | Final = 0.7958
Rank 5 : movie_0347 | Semantic = 0.7854 | BPR = -45.0072 | Final = 0.7904
Rank 6 : movie_0369 | Semantic = 0.7691 | BPR = -44.7874 | Final = 0.7843
Rank 7 : movie_0822 | Semantic = 0.7921 | BPR = -45.6943 | Final = 0.7785
Rank 8 : movie_0460 | Semantic = 0.8138 | BPR = -46.3808 | Final = 0.7772
Rank 9 : movie_0879 | Semantic = 0.7702 | BPR = -45.2185 | Final = 0.7747
Rank 10 : movie_0638 | Semantic = 0.7817 | BPR = -45.6204 | Final = 0.773


In [ ]:
def get_movie_information(
    movie_id
):

    row = movie_items[
        movie_items["id"] == movie_id
    ]

    if len(row) == 0:
        return None

    row = row.iloc[0]

    return {
        "movie_id": movie_id,
        "title": row["title"],
        "domain": row["domain"],
        "metadata": row["metadata"]
    }

In [ ]:
results = []

for rank in range(
    len(top_10)
):

    item = top_10[rank]

    movie_info = get_movie_information(
        item["movie_id"]
    )

    results.append({
        "rank": rank + 1,
        "movie_id": item["movie_id"],
        "title": movie_info["title"],
        "domain": movie_info["domain"],
        "metadata": movie_info["metadata"],
        "semantic_score": item["semantic_score"],
        "bpr_score": item["bpr_score"],
        "final_score": item["final_score"]
    })

sample_result = pd.DataFrame(
    results
)

sample_result

,rank,movie_id,title,domain,metadata,semantic_score,bpr_score,final_score
0,1,movie_0534,big family,tv series,war sport,0.768897,-42.542656,0.838228
1,2,movie_0103,family battle,stand-up comedy,sci-fi action,0.809123,-45.161934,0.803295
2,3,movie_0348,quest adventure,tv series,sport family,0.787296,-44.619102,0.801092
3,4,movie_0681,a family,stand-up comedy,thriller,0.782498,-44.701027,0.795760
4,5,movie_0347,story warrior,tv series,comedy family,0.785391,-45.007198,0.790410
5,6,movie_0369,big ice,documentary,family adventure,0.769055,-44.787388,0.784270
6,7,movie_0822,family mystery,tv series,war family,0.792084,-45.694267,0.778546
7,8,movie_0460,family journey,tv series,documentary history,0.813771,-46.380821,0.777190
8,9,movie_0879,storm fire,tv series,action family,0.770195,-45.218475,0.774684
9,10,movie_0638,quest family,tv series,romance,0.781669,-45.620361,0.773035


In [ ]:
all_recommendations = []

for count, user_id in enumerate(
    test_users
):

    if user_id not in user_embeddings:
        continue

    candidates = retrieve_candidates_with_scores(
        user_id,
        candidate_k=500
    )

    candidates = add_bpr_scores(
        user_id,
        candidates
    )

    candidates = combine_scores(
        candidates
    )

    top_10 = rerank_candidates(
        candidates,
        top_k=10
    )

    for rank in range(
        len(top_10)
    ):

        item = top_10[rank]

        movie_info = get_movie_information(
            item["movie_id"]
        )

        all_recommendations.append({
            "user_id": user_id,
            "rank": rank + 1,
            "movie_id": item["movie_id"],
            "title": movie_info["title"],
            "domain": movie_info["domain"],
            "metadata": movie_info["metadata"],
            "semantic_score": item["semantic_score"],
            "bpr_score": item["bpr_score"],
            "final_score": item["final_score"]
        })

    if count % 500 == 0:

        print(
            "Processed:",
            count
        )

Processed: 0
Processed: 500
Processed: 1000
Processed: 1500
Processed: 2000
Processed: 2500
Processed: 3000
Processed: 3500
Processed: 4000
Processed: 4500
Processed: 5000
Processed: 5500
Processed: 6000
Processed: 6500
Processed: 7000
Processed: 7500
Processed: 8000
Processed: 8500


In [ ]:
phase3_recommendations = pd.DataFrame(
    all_recommendations
)

print(
    "Phase 3 recommendation shape:",
    phase3_recommendations.shape
)

print(
    phase3_recommendations.head(20)
)

Phase 3 recommendation shape: (100000, 9)
       user_id  rank    movie_id           title           domain  \
0   user_01838     1  movie_0572    dream secret        tv series   
1   user_01838     2  movie_0497     big warrior        tv series   
2   user_01838     3  movie_0951       big house        tv series   
3   user_01838     4  movie_0186       old house        tv series   
4   user_01838     5  movie_0059    night secret        tv series   
5   user_01838     6  movie_0200   legend secret        tv series   
6   user_01838     7  movie_0902     mystery ice        tv series   
7   user_01838     8  movie_0845    little house        tv series   
8   user_01838     9  movie_0882  little mystery        tv series   
9   user_01838    10  movie_0718   mystery dream        tv series   
10  user_06815     1  movie_0448      story hero            movie   
11  user_06815     2  movie_0087       our storm            movie   
12  user_06815     3  movie_0243       new quest            m

In [ ]:
sample_user = test_users[0] if isinstance(
    test_users,
    list
) else list(test_users)[0]

user_recommendations = (
    phase3_recommendations[
        phase3_recommendations["user_id"]
        == sample_user
    ]
    .sort_values("rank")
)

user_recommendations

,user_id,rank,movie_id,title,domain,metadata,semantic_score,bpr_score,final_score
0,user_01838,1,movie_0572,dream secret,tv series,documentary sci-fi,0.839006,0.411912,0.832234
1,user_01838,2,movie_0497,big warrior,tv series,family,0.766900,0.615895,0.818021
2,user_01838,3,movie_0951,big house,tv series,horror,0.862037,0.170196,0.805388
3,user_01838,4,movie_0186,old house,tv series,comedy,0.791097,0.446407,0.804830
4,user_01838,5,movie_0059,night secret,tv series,romance sci-fi,0.821137,0.320560,0.803487
5,user_01838,6,movie_0200,legend secret,tv series,drama,0.799843,0.384161,0.799887
6,user_01838,7,movie_0902,mystery ice,tv series,action drama,0.749005,0.560189,0.795592
7,user_01838,8,movie_0845,little house,tv series,thriller,0.796414,0.350598,0.791521
8,user_01838,9,movie_0882,little mystery,tv series,horror,0.805351,0.286914,0.786456
9,user_01838,10,movie_0718,mystery dream,tv series,sci-fi,0.862476,0.061773,0.786422


In [ ]:
phase3_recommendation_file = os.path.join(
    PHASE3_PATH,
    "phase3_recommendations.csv"
)

phase3_recommendations.to_csv(
    phase3_recommendation_file,
    index=False
)

print(
    "Saved:",
    phase3_recommendation_file
)

Saved: C:\nihal\rough\Reccomender_system_rough\Phase_3\phase3_recommendations.csv


In [ ]:
actual_test = test_interactions[
    ["user_id", "item_id"]
].copy()

actual_test = actual_test.rename(
    columns={
        "item_id": "actual_movie_id"
    }
)

print(actual_test.head())

      user_id actual_movie_id
0  user_00001      movie_0483
1  user_00002      movie_0362
2  user_00003      movie_0855
3  user_00004      movie_0712
4  user_00005      movie_0423


In [ ]:
phase3_eval = phase3_recommendations.merge(
    actual_test,
    on="user_id",
    how="left"
)

print(
    "Evaluation shape:",
    phase3_eval.shape
)

Evaluation shape: (100000, 10)


In [ ]:
phase3_eval["exact_match"] = (
    phase3_eval["movie_id"]
    ==
    phase3_eval["actual_movie_id"]
)

print(
    "Exact matches:",
    phase3_eval["exact_match"].sum()
)

Exact matches: 112


In [ ]:
users = phase3_eval[
    "user_id"
].unique()

precision_values = []

for user_id in users:

    user_rows = phase3_eval[
        phase3_eval["user_id"] == user_id
    ]

    hits = user_rows[
        "exact_match"
    ].sum()

    precision = hits / 10

    precision_values.append(
        precision
    )

precision_at_10 = np.mean(
    precision_values
)

print(
    "Phase 3 Precision@10:",
    precision_at_10
)

Phase 3 Precision@10: 0.0011200000000000003


In [ ]:
recall_values = []

for user_id in users:

    user_rows = phase3_eval[
        phase3_eval["user_id"] == user_id
    ]

    if user_rows[
        "exact_match"
    ].any():

        recall_values.append(1)

    else:

        recall_values.append(0)

recall_at_10 = np.mean(
    recall_values
)

print(
    "Phase 3 Recall@10:",
    recall_at_10
)

Phase 3 Recall@10: 0.0112


In [ ]:
rr_values = []

for user_id in users:

    user_rows = phase3_eval[
        phase3_eval["user_id"] == user_id
    ].sort_values("rank")

    reciprocal_rank = 0

    for _, row in user_rows.iterrows():

        if row["exact_match"]:

            reciprocal_rank = (
                1 / row["rank"]
            )

            break

    rr_values.append(
        reciprocal_rank
    )

mrr = np.mean(
    rr_values
)

print(
    "Phase 3 MRR:",
    mrr
)

Phase 3 MRR: 0.0033137301587301588


In [ ]:
def get_semantic_similarity(
    movie_id_1,
    movie_id_2
):

    if movie_id_1 not in movie_id_to_idx:
        return 0.0

    if movie_id_2 not in movie_id_to_idx:
        return 0.0

    idx1 = movie_id_to_idx[
        movie_id_1
    ]

    idx2 = movie_id_to_idx[
        movie_id_2
    ]

    vector1 = movie_embeddings[
        idx1
    ]

    vector2 = movie_embeddings[
        idx2
    ]

    score = torch.dot(
        vector1,
        vector2
    )

    return score.item()

In [ ]:
similarities = []

for _, row in phase3_eval.iterrows():

    score = get_semantic_similarity(
        row["movie_id"],
        row["actual_movie_id"]
    )

    similarities.append(
        score
    )

phase3_eval[
    "similarity_to_actual"
] = similarities

In [ ]:
SEMANTIC_THRESHOLD = 0.80

phase3_eval[
    "semantic_match"
] = (
    phase3_eval[
        "similarity_to_actual"
    ]
    >= SEMANTIC_THRESHOLD
)

print(
    phase3_eval[
        "semantic_match"
    ].mean()
)

0.07837


In [ ]:
semantic_recall_values = []

for user_id in users:

    user_rows = phase3_eval[
        phase3_eval["user_id"] == user_id
    ]

    if user_rows[
        "semantic_match"
    ].any():

        semantic_recall_values.append(1)

    else:

        semantic_recall_values.append(0)

semantic_recall_at_10 = np.mean(
    semantic_recall_values
)

print(
    "Semantic Recall@10:",
    semantic_recall_at_10
)

Semantic Recall@10: 0.3615


In [ ]:
semantic_rr_values = []

for user_id in users:

    user_rows = phase3_eval[
        phase3_eval["user_id"] == user_id
    ].sort_values("rank")

    reciprocal_rank = 0

    for _, row in user_rows.iterrows():

        if row["semantic_match"]:

            reciprocal_rank = (
                1 / row["rank"]
            )

            break

    semantic_rr_values.append(
        reciprocal_rank
    )

semantic_mrr = np.mean(
    semantic_rr_values
)

print(
    "Semantic MRR:",
    semantic_mrr
)

Semantic MRR: 0.15518174603174603


In [ ]:
def calculate_ndcg(
    user_rows,
    relevance_column
):

    relevance = user_rows[
        relevance_column
    ].astype(float).tolist()

    dcg = 0.0

    for i in range(
        len(relevance)
    ):

        rank = i + 1

        dcg += (
            relevance[i] /
            np.log2(rank + 1)
        )

    ideal = sorted(
        relevance,
        reverse=True
    )

    idcg = 0.0

    for i in range(
        len(ideal)
    ):

        rank = i + 1

        idcg += (
            ideal[i] /
            np.log2(rank + 1)
        )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [ ]:
ndcg_values = []

for user_id in users:

    user_rows = phase3_eval[
        phase3_eval["user_id"] == user_id
    ].sort_values("rank")

    score = calculate_ndcg(
        user_rows,
        "exact_match"
    )

    ndcg_values.append(
        score
    )

ndcg_at_10 = np.mean(
    ndcg_values
)

print(
    "Phase 3 NDCG@10:",
    ndcg_at_10
)

Phase 3 NDCG@10: 0.005108210331194963


In [ ]:
semantic_ndcg_values = []

for user_id in users:

    user_rows = phase3_eval[
        phase3_eval["user_id"] == user_id
    ].sort_values("rank")

    relevance = user_rows[
        "similarity_to_actual"
    ].astype(float).tolist()

    dcg = 0.0

    for i in range(
        len(relevance)
    ):

        rank = i + 1

        dcg += (
            relevance[i] /
            np.log2(rank + 1)
        )

    ideal = sorted(
        relevance,
        reverse=True
    )

    idcg = 0.0

    for i in range(
        len(ideal)
    ):

        rank = i + 1

        idcg += (
            ideal[i] /
            np.log2(rank + 1)
        )

    if idcg == 0:

        score = 0.0

    else:

        score = dcg / idcg

    semantic_ndcg_values.append(
        score
    )

semantic_ndcg_at_10 = np.mean(
    semantic_ndcg_values
)

print(
    "Semantic NDCG@10:",
    semantic_ndcg_at_10
)

Semantic NDCG@10: 0.9699955849379117


In [ ]:
phase3_metrics = {
    "Precision@10":
        precision_at_10,

    "Recall@10":
        recall_at_10,

    "MRR":
        mrr,

    "NDCG@10":
        ndcg_at_10,

    "SemanticRecall@10":
        semantic_recall_at_10,

    "SemanticMRR":
        semantic_mrr,

    "SemanticNDCG@10":
        semantic_ndcg_at_10
}

phase3_metrics_df = pd.DataFrame(
    [phase3_metrics]
)

phase3_metrics_df

,Precision@10,Recall@10,MRR,NDCG@10,SemanticRecall@10,SemanticMRR,SemanticNDCG@10
0,0.00112,0.0112,0.003314,0.005108,0.3615,0.155182,0.969996


In [ ]:
phase2_metrics = pd.read_csv(
    os.path.join(
        PHASE2_PATH,
        "phase2_metrics.csv"
    )
)

print(
    phase2_metrics
)

                               model  Precision@10  Recall@10       MRR  \
0  Phase 2 - Weighted User Embedding       0.00111     0.0111  0.003317   

    NDCG@10  SemanticRecall@10  SemanticMRR  SemanticNDCG@10  
0  0.005111             0.2909     0.138038         0.975796  


In [ ]:
phase2_row = phase2_metrics.iloc[0]

comparison = pd.DataFrame({
    "Metric": [
        "Precision@10",
        "Recall@10",
        "MRR",
        "NDCG@10",
        "SemanticRecall@10",
        "SemanticMRR",
        "SemanticNDCG@10"
    ],

    "Phase 2": [
        phase2_row["Precision@10"],
        phase2_row["Recall@10"],
        phase2_row["MRR"],
        phase2_row["NDCG@10"],
        phase2_row["SemanticRecall@10"],
        phase2_row["SemanticMRR"],
        phase2_row["SemanticNDCG@10"]
    ],

    "Phase 3": [
        precision_at_10,
        recall_at_10,
        mrr,
        ndcg_at_10,
        semantic_recall_at_10,
        semantic_mrr,
        semantic_ndcg_at_10
    ]
})

comparison

,Metric,Phase 2,Phase 3
0,Precision@10,0.001110,0.001120
1,Recall@10,0.011100,0.011200
2,MRR,0.003317,0.003314
3,NDCG@10,0.005111,0.005108
4,SemanticRecall@10,0.290900,0.361500
5,SemanticMRR,0.138038,0.155182
6,SemanticNDCG@10,0.975796,0.969996


In [ ]:
comparison["Improvement"] = (
    comparison["Phase 3"]
    -
    comparison["Phase 2"]
)

comparison["Improvement %"] = np.where(
    comparison["Phase 2"] != 0,

    (
        comparison["Improvement"]
        /
        comparison["Phase 2"]
    ) * 100,

    0
)

comparison

,Metric,Phase 2,Phase 3,Improvement,Improvement %
0,Precision@10,0.001110,0.001120,0.000010,0.900901
1,Recall@10,0.011100,0.011200,0.000100,0.900901
2,MRR,0.003317,0.003314,-0.000004,-0.107661
3,NDCG@10,0.005111,0.005108,-0.000003,-0.052051
4,SemanticRecall@10,0.290900,0.361500,0.070600,24.269508
5,SemanticMRR,0.138038,0.155182,0.017143,12.419378
6,SemanticNDCG@10,0.975796,0.969996,-0.005800,-0.594394


In [ ]:
phase3_metrics_df.to_csv(
    os.path.join(
        PHASE3_PATH,
        "phase3_metrics.csv"
    ),
    index=False
)

comparison.to_csv(
    os.path.join(
        PHASE3_PATH,
        "phase2_vs_phase3.csv"
    ),
    index=False
)

print(
    "Final metrics saved."
)

Final metrics saved.


In [ ]:
phase3_eval.to_csv(
    os.path.join(
        PHASE3_PATH,
        "phase3_detailed_evaluation.csv"
    ),
    index=False
)

print(
    "Detailed evaluation saved."
)

Detailed evaluation saved.


In [ ]:
# torch.save(
#     bpr_model.state_dict(),
#     os.path.join(
#         PHASE3_PATH,
#         "bpr_model.pt"
#     )
# )

# print(
#     "BPR model saved."
# )

BPR model saved.
